# Chapter 7 · 금융 누수·Walk-forward

**연구질문:** 누수(스케일링·미래 정보·무작위 분할)는 성능을 얼마나 과대평가하는가? Walk-forward에서 결론은 유지되는가?

> 위에서 아래로 실행하면 끝까지 동작합니다.

## 1. 핵심 개념·수식

**누수(leakage):** 예측 시점 $t$에 알 수 없는 정보가 특성·전처리·분할에 들어가 성능이 과대평가되는 문제. 대표 경로 세 가지.
1. **전처리 누수:** 표준화·결측대체·모수 추정을 전체 자료로 한다(학습구간 통계만 써야 함).
2. **미래 정보 누수:** 중심 이동평균 등 $y_{t+k}$를 포함하는 특성.
3. **분할 누수:** 시계열에 무작위 K-겹 교차검증을 쓴다(인접 관측이 학습·검증에 섞임).

**Walk-forward(롤링 오리진) 평가:** 시점 $T_1<T_2<\dots$마다 $[1,T_k]$로 학습하고 $(T_k,T_k+h]$를 평가한다. 폴드별 오차의 분포(평균·분산·최악)를 본다.

**Diebold–Mariano 검정:** 손실차 $d_t=L(e^{A}_t)-L(e^{B}_t)$의 평균이 0인지 검정한다. $\mathrm{DM}=\bar d/\sqrt{\widehat{\mathrm{LRV}}(d)/n}$.

**부트스트랩 신뢰구간:** 폴드 손실차를 블록 부트스트랩하여 우열의 불확실성을 본다.

In [ ]:
import os, io, json, subprocess, sys, warnings, urllib.request, urllib.parse
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, optimize

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
SEED = 42
# 네트워크 없이 합성자료로만 실행하려면 환경변수 TSLAB_OFFLINE=1 로 실행하거나 아래를 False로 바꾼다.
USE_LIVE_DATA = os.environ.get("TSLAB_OFFLINE") != "1"
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True,
                     "axes.spines.top": False, "axes.spines.right": False})

def ensure(pkg, mod=None):
    # 없으면 설치를 시도한다. 실패해도 노트북은 계속 실행되고 해당 선택 기능만 건너뛴다.
    try:
        __import__(mod or pkg); return True
    except Exception:
        if not USE_LIVE_DATA:
            return False
        try:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True, timeout=300)
            __import__(mod or pkg); return True
        except Exception as e:
            print(f"[건너뜀] {pkg} 설치/가져오기 실패: {type(e).__name__}")
            return False

def fetch_bytes(url, timeout=60):
    req = urllib.request.Request(url, headers={"User-Agent": "ts-lab-course/1.0 (education)"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()

# ---- 평가 지표와 기준선 ----
def mae(a, f):  return float(np.mean(np.abs(np.asarray(a, float) - np.asarray(f, float))))
def rmse(a, f): return float(np.sqrt(np.mean((np.asarray(a, float) - np.asarray(f, float)) ** 2)))
def mase(a, f, train, m=1):
    train = np.asarray(train, float)
    scale = np.mean(np.abs(train[m:] - train[:-m]))
    return mae(a, f) / scale
def naive_fc(train, h):  return np.repeat(np.asarray(train, float)[-1], h)
def snaive_fc(train, h, m):
    train = np.asarray(train, float)
    return np.tile(train[-m:], int(np.ceil(h / m)))[:h]
def drift_fc(train, h):
    train = np.asarray(train, float)
    slope = (train[-1] - train[0]) / (len(train) - 1)
    return train[-1] + slope * np.arange(1, h + 1)
def show(df, digits=4):
    print(df.round(digits).to_string())

In [ ]:
def garch_sim(n, seed, omega=0.02, alpha=0.08, beta=0.90):
    rng = np.random.default_rng(seed); r = np.zeros(n); s2 = omega / (1 - alpha - beta)
    for t in range(n):
        r[t] = np.sqrt(s2) * rng.standard_normal()
        s2 = omega + alpha * r[t] ** 2 + beta * s2
    return r + 0.04                                            # 일수익률(%) , 약간의 양의 평균

def load_finance():
    px, kind = None, "SYNTHETIC (NOT real data): GARCH(1,1) 수익률로 만든 가격"
    if USE_LIVE_DATA and ensure("yfinance"):
        try:
            import yfinance as yf
            raw = yf.download("SPY", start="2015-01-01", progress=False, auto_adjust=True)
            c = raw["Close"]
            if isinstance(c, pd.DataFrame):
                c = c.iloc[:, 0]
            c = c.dropna().astype(float)
            if len(c) > 500:
                px, kind = c, "REAL: SPY (Yahoo Finance, yfinance)"
        except Exception as e:
            print("온라인 수집 실패 -> 합성 대체:", type(e).__name__, str(e)[:120])
    if px is None:
        r = garch_sim(2500, SEED)
        px = pd.Series(100 * np.exp(np.cumsum(r / 100)), index=pd.bdate_range("2015-01-01", periods=2500))
    px.index = pd.to_datetime(px.index)
    if getattr(px.index, "tz", None) is not None:
        px.index = px.index.tz_localize(None)
    return dict(y=px.rename("price"), kind=kind, m=5, freq="B", label="SPY 종가", unit="USD")

D = load_finance()
y, DATA_KIND, M, FREQ = D["y"], D["kind"], D["m"], D["freq"]
ret = (np.log(y).diff().dropna() * 100).rename("log_return_pct")      # 로그수익률(%)
print("DATA_KIND:", DATA_KIND); print("관측 수:", len(y), "| 기간:", y.index.min().date(), "~", y.index.max().date())

## 2. Walk-forward 평가

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold

def make_X(s, lags=(1, 2, 3, 5, 10), rolls=(5, 20)):
    X = pd.DataFrame({f"lag_{k}": s.shift(k) for k in lags})
    for w in rolls:
        X[f"rm_{w}"] = s.shift(1).rolling(w).mean(); X[f"rs_{w}"] = s.shift(1).rolling(w).std()
    return X

def dm_stat(d, h=1):
    d = np.asarray(d, float); n = len(d); dbar = d.mean(); var = np.var(d)
    for l in range(1, max(h, 1)):
        var += 2 * (1 - l / h) * np.mean((d[l:] - dbar) * (d[:-l] - dbar))
    stat = dbar / np.sqrt(var / n); return stat, 2 * (1 - stats.norm.cdf(abs(stat)))

def block_bootstrap_ci(d, block=5, B=1000, seed=SEED):
    rng = np.random.default_rng(seed); d = np.asarray(d, float); n = len(d); out = []
    for _ in range(B):
        starts = rng.integers(0, max(n - block + 1, 1), size=int(np.ceil(n / block)))
        out.append(np.concatenate([d[s0:s0 + block] for s0 in starts])[:n].mean())
    return np.percentile(out, [2.5, 97.5])

In [ ]:
r = ret.astype(float); X = make_X(r); data = pd.concat([X, r.rename("target")], axis=1).dropna()
XX, yy = data.drop(columns="target"), data["target"]
def models():
    return {"Ridge": make_pipeline(StandardScaler(), Ridge(alpha=10.0)),
            "HistGB": HistGradientBoostingRegressor(max_iter=150, learning_rate=0.05, random_state=SEED)}

# ---- A. 올바른 평가: Walk-forward (확장 창, 폴드 8개) ----
n = len(data); n_folds = 8; fold = max(100, n // 12); start = n - n_folds * fold
rows, loss = [], {"hist_mean": [], "Ridge": [], "HistGB": []}
for k in range(n_folds):
    a, b = start + k * fold, start + (k + 1) * fold
    Xtr, ytr, Xte, yte = XX.iloc[:a], yy.iloc[:a], XX.iloc[a:b], yy.iloc[a:b]
    base = np.full(len(yte), ytr.mean()); loss["hist_mean"].append(np.mean((yte - base) ** 2))
    for name, m in models().items():
        m.fit(Xtr, ytr); loss[name].append(np.mean((yte.to_numpy() - m.predict(Xte)) ** 2))
L = pd.DataFrame(loss, index=[f"fold{k+1}" for k in range(n_folds)])
print("폴드별 MSE:"); show(L)
print("평균 MSE:"); show(L.mean().to_frame("MSE"))
for name in ("Ridge", "HistGB"):
    d = L["hist_mean"] - L[name]            # 양수면 모델이 역사평균보다 낫다
    st, pv = dm_stat(d.to_numpy()); lo, hi = block_bootstrap_ci(d.to_numpy(), block=2)
    print(f"{name} vs 역사평균: 평균 손실개선={d.mean():.4f}, DM p≈{pv:.3f} (폴드 {len(d)}개, 근사), 부트스트랩 95% CI=[{lo:.4f},{hi:.4f}]")
wf_mse = float(L["Ridge"].mean())

## 3. 누수 실험

In [ ]:
# ---- B. 누수를 일부러 만들어 성능이 얼마나 부풀려지는지 확인 ----
split = int(len(data) * 0.8)
def holdout(Xa, ya, scale_on_all=False, model=None):
    Xa = Xa.copy()
    mu_sd = (Xa.mean(), Xa.std()) if scale_on_all else (Xa.iloc[:split].mean(), Xa.iloc[:split].std())   # 누수 1: 전체 통계로 표준화
    Z = (Xa - mu_sd[0]) / mu_sd[1]
    m = model or Ridge(alpha=10.0); m.fit(Z.iloc[:split], ya.iloc[:split]); p = m.predict(Z.iloc[split:])
    return float(np.mean((ya.iloc[split:].to_numpy() - p) ** 2))
res_leak = {"정상(학습구간 통계로 표준화)": holdout(XX, yy)}
res_leak["누수1: 전체 자료로 표준화"] = holdout(XX, yy, scale_on_all=True)
# 누수 2: 중심 이동평균 특성(미래 포함)
Xc = XX.copy(); Xc["centered_ma"] = r.rolling(5, center=True).mean().loc[XX.index]; Xc = Xc.dropna(); yc = yy.loc[Xc.index]
split_c = int(len(Xc) * 0.8)
m2 = Ridge(alpha=1.0).fit(Xc.iloc[:split_c], yc.iloc[:split_c])
res_leak["누수2: 중심 이동평균(미래 포함)"] = float(np.mean((yc.iloc[split_c:].to_numpy() - m2.predict(Xc.iloc[split_c:])) ** 2))
# 누수 3: 무작위 K-겹 (시간 섞임)
kf = KFold(n_splits=5, shuffle=True, random_state=SEED); errs = []
for tr_i, te_i in kf.split(XX):
    m3 = HistGradientBoostingRegressor(max_iter=150, learning_rate=0.05, random_state=SEED).fit(XX.iloc[tr_i], yy.iloc[tr_i])
    errs.append(np.mean((yy.iloc[te_i].to_numpy() - m3.predict(XX.iloc[te_i])) ** 2))
res_leak["누수3: 무작위 K-겹 CV (HistGB)"] = float(np.mean(errs))
res_leak["참고: 역사평균 기준선"] = float(np.mean((yy.iloc[split:].to_numpy() - yy.iloc[:split].mean()) ** 2))
tabL = pd.Series(res_leak).to_frame("MSE"); show(tabL)
print("해석 가이드: 누수 설정의 MSE가 정상 설정보다 유의하게 작다면 그 차이가 '누수가 만든 허위 성능'이다. 수익률은 예측이 어려워 차이가 작을 수 있다 -> 누수의 영향이 큰 자료는 이후 도메인(센서·프로모션)에서 확인.")

## 3. 결과 해석 작성
1. 정상 설정과 누수 설정의 성능 차이를 표로 제시하고, **어떤 누수 경로가 가장 큰 왜곡을 만들었는지** 설명한다.
2. 폴드(또는 지역·기간) 간 변동을 보고 "평균은 좋지만 최악의 경우가 나쁘다"를 구분한다.
3. 결론(모형 우열, 효과 방향)이 설정 변경에도 유지되는가? 유지되지 않으면 "강건하다"가 아니라 "민감하다"로 쓴다.
4. 한계: 폴드 수, 이벤트 수, 합성자료의 한계, 위약검정이 모든 식별 가정을 증명하지는 않는다는 점.

**제출:** 정상/누수 비교표, 폴드별 결과, 강건성 표, 한계 단락. 누수 점검 목록(`validation/Leakage_Checklist.md`)과 대조하라.